In [ ]:
### Cu 003 processing ###


#%% load the packages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.ticker import FuncFormatter
from matplotlib import cm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import defdap.hrdic as hrdic
import defdap.ebsd as ebsd
import defdap.experiment as experiment
from defdap.quat import Quat
from defdap.plotting import MapPlot

from pathlib import Path

import copy 
import pandas as pd
import datetime

from scipy.signal import find_peaks
from scipy.linalg import sqrtm, polar
from scipy import stats
from scipy import interpolate 


import skimage as ski


import os

# get dictools stuff 
import sys
# sys.path.append("c:/work/hrdic-tools/")
# import dictools

plt.rcParams['svg.fonttype'] = 'none'

%matplotlib qt

In [2]:
def calc_rotations(dic_map):
    # calculate rotations from dic displacement field 

    # extract deformation gradient
    f = dic_map.data.f

    # calculate rotation as ang = (F21 - F12)/2
    rot = (f[0,1,:,:] - f[1,0,:,:])/2

    # centre on mean 
    rot = rot - np.nanmean(rot)

    return rot 

In [3]:
exp = experiment.Experiment()

# load DIC data 
data_dir = Path('./DIC/pyvale/')
dic_frame = experiment.Frame()

# dic_step_list = sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv'))

for dic_file in sorted(data_dir.glob('dic_output_fixed_Step_*_subset_31_step_10.csv')):
    hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

# dic_file = dic_step_list[-2]
# hrdic.Map(dic_file, experiment=exp, frame=dic_frame,data_type='pyvale-csv')

hfw = 20.0 # microns
pixelwidth = 2048
pixelsize = hfw/pixelwidth



# calculate rotations    
for inc, dic_map in exp.iter_over_maps('hrdic'):
    # add rotation map to dic

    # calculate rotation
    rot = calc_rotations(dic_map)*180/np.pi

    # add to dic_map
    dic_map.data.add(
        'r_ang', rot,
        unit='°', type='map', order=0,
        plot_params={
            'plot_colour_bar': True,
            'clabel': 'Rotation',
            'cmap': 'RdBu_r'
        }
    )



for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.set_scale(pixelsize)
    dic_map.set_crop(left=100,right=100,top=100,bottom=100)
    # dic_map.plot_map('max_shear',vmin=0,vmax=0.01,plot_scale_bar=True)
    print(dic_map)

Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a data (dimensions: 3197 x 3197 pixels, sub-window size: 10 x 10 pixels)
Loaded PyVale n/a dat

In [4]:
ebsd_frame = experiment.Frame()
data_dir = Path('.')
ebsd.Map(data_dir / 'Pre_EBSD/map.cpr',
         increment=exp.increments[0], frame=ebsd_frame)

Loaded EBSD data (dimensions: 3727 x 2795 pixels, step size: 0.2 um)


In [5]:
ebsd_map = exp.increments[0].maps['ebsd']
# ebsd_map.set_homog_point()

dic_map = exp.increments[0].maps['hrdic']

# dic_map.set_homog_point(vmin=0,vmax=0.05)

In [6]:
ebsd_frame.homog_points = [(1946, 1565),
 (2443, 1000),
 (1305, 1027),
 (1395, 2225),
 (2572, 2193),
 (1876, 1077),
 (2613, 1497),
 (1822, 2208),
 (1259, 1661),
 (2229, 1260),
 (1641, 1325),
 (1693, 1794),
 (2195, 1762)]

In [7]:
dic_frame.homog_points = [(1582, 1380),
 (2615, 172),
 (238, 226),
 (453, 2748),
 (2882, 2728),
 (1431, 326),
 (2970, 1240),
 (1329, 2732),
 (157, 1568),
 (2170, 731),
 (941, 863),
 (1061, 1854),
 (2100, 1795)]

In [8]:
ebsd_map = exp.increments[0].maps['ebsd']

for inc, dic_map in exp.iter_over_maps('hrdic'):
    dic_map.link_ebsd_map(ebsd_map, transform_type="polynomial",order=2)
    # dic_map.link_ebsd_map(ebsd_map, transform_type="affine")

In [ ]:
# Figure 2 rotation plot

# plot = MapPlot.create(dic_map,rot,rot,vmin=-5,vmax=5,cmap='RdBu_r',
#                       plot_gbs='pixel',
#                       dilate_boundaries=True,
#                       boundary_colour='black',
#                       plot_colour_bar=True,
#                       clabel = 'In-plane rotation / °',
#                       plot_scale_bar=True                     
#                       )
# plt.savefig('./figures_for_paper/dic_rotation_max_load_with_gbs.png',dpi=1000)


In [ ]:
# Figure 2 ipf - x plot 

# ebsd_map.plot_map('orientation',component='IPF_x',plot_gbs='pixel',dilate_boundaries=True,plot_scale_bar=True)#,extents=(2000,6000,1000,2500))
# ax=plt.gca()
# ax.set_xlim([1221,2620])
# ax.set_ylim([2327,930])

# plt.savefig('./figures_for_paper/ipfx_with_gbs.png',dpi=1000)

In [ ]:
# Figure 2 max shear plot 

# dic_map.plot_map('max_shear',vmin=0,vmax=0.1,plot_gbs='line',plot_scale_bar=True,dilate_boundaries=True)
# plt.savefig('./figures_for_paper/max_shear_max_load_with_gbs.png',dpi=1000)

In [12]:
# find special boundaries 
misori_twin = Quat.from_axis_angle([1, 1, 1], 60*np.pi/180)
misori_twin_tol = 10*np.pi/180

# create all symmetric equivalent misorientations
misori_twin_all = []
syms = ebsd_map.primary_phase.crystal_structure.symmetries
for sym_i in syms:
    for sym_j in syms:
        misori_twin_all.append(sym_i.conjugate * misori_twin * sym_j)


# get rid of any duplicates
misori_twin_all = list(set(misori_twin_all))

# calculate neighbour network
ebsd_map.build_neighbour_network()

# loop over all grain boundary segments and check if the misorientation between
# the two grains is within tolerance of the twin misorientation
# store all gbs pairs with misorientation for later us 

all_gb_info = []
twin_gb_info = []
twin_lines = []

for grain1, grain2, b_seg in ebsd_map.neighbour_network.edges.data('boundary'):
    twin = False

    # calculate grain ref orientation
    grain1.calc_average_ori()
    grain2.calc_average_ori()

    misori = grain2.ref_ori * grain1.ref_ori.conjugate

    # # calculate misorientation angle in degrees 
    misori_ang = 2*np.arccos(grain1.ref_ori.mis_ori(grain2.ref_ori,ebsd_map.crystal_sym))*180/np.pi

    # add to list of everything 
    all_gb_info.append([grain1,grain2,misori,misori_ang,b_seg])

    # check if twin and add to twin list
    for misori_twin in misori_twin_all:
        if 2 * np.arccos(misori_twin.dot(misori)) < misori_twin_tol:
            twin = True
            break
    
    if not twin:
        continue
    
    twin_lines.append(b_seg)
    twin_gb_info.append([grain1,grain2,misori,misori_ang,b_seg])




# make into boundary set
s_bounds = ebsd.BoundarySet.from_boundary_segments(twin_lines)

Finished finding grains (0:00:15) twork..
Finished constructing neighbour network (0:00:21) 


In [13]:
# data cubifier 
# make the individual maps into a data cube nx * ny * n_DIC_steps 

def datacubifier(exp):
    # find number of steps 
    n_steps = len(exp.increments)

    # get data shape 
    ny,nx = exp.increments[0].maps['hrdic'].shape

    # make empty arrays to fill things with 
    ems_cube    = np.zeros((ny,nx,n_steps))
    e11_cube    = np.zeros((ny,nx,n_steps))
    e22_cube    = np.zeros((ny,nx,n_steps))
    e12_cube    = np.zeros((ny,nx,n_steps))
    r_ang_cube  = np.zeros((ny,nx,n_steps))
    u_cube      = np.zeros((ny,nx,n_steps))
    v_cube      = np.zeros((ny,nx,n_steps))
    f11_cube    = np.zeros((ny,nx,n_steps))
    f12_cube    = np.zeros((ny,nx,n_steps))
    f21_cube    = np.zeros((ny,nx,n_steps))
    f22_cube    = np.zeros((ny,nx,n_steps))

    for inc, dic_map in exp.iter_over_maps('hrdic'):
        ems_cube[:,:,inc]   = dic_map.data['max_shear']
        e11_cube[:,:,inc]   = dic_map.data['e'][0,0]
        e22_cube[:,:,inc]   = dic_map.data['e'][1,1]
        e12_cube[:,:,inc]   = dic_map.data['e'][0,1]
        r_ang_cube[:,:,inc] = dic_map.data['r_ang']
        u_cube[:,:,inc]     = dic_map.data['displacement'][0]
        v_cube[:,:,inc]     = dic_map.data['displacement'][1]
        f11_cube[:,:,inc]   = dic_map.data['f'][0,0]
        f12_cube[:,:,inc]   = dic_map.data['f'][0,1]
        f21_cube[:,:,inc]   = dic_map.data['f'][1,0]
        f22_cube[:,:,inc]   = dic_map.data['f'][1,1]
        



    return ems_cube, e11_cube, e22_cube, e12_cube, r_ang_cube, u_cube, v_cube, f11_cube, f12_cube, f21_cube, f22_cube

# cubify data for more simple processing
ems_cube, e11_cube, e22_cube, e12_cube, r_ang_cube, u_cube, v_cube, f11_cube, f12_cube, f21_cube, f22_cube = datacubifier(exp)
e11_glob = np.nanmean(e11_cube,axis=(0,1))

In [21]:
# plot misorientation distribution function 

# sanity check for all twin boundaries having a MO ~ 60 degrees 
twin_mo_angle = []
for i in range(len(twin_gb_info)):
    twin_mo_angle.append(twin_gb_info[i][3])



all_mo_angle = []
for i in range(len(all_gb_info)):
    all_mo_angle.append(all_gb_info[i][3])

fig,ax = plt.subplots()
ax.hist(all_mo_angle,range=(0,65),bins=50,density=True,color='yellowgreen',edgecolor='olivedrab')
ax.set_ylabel('Probability density / -')
ax.set_xlabel('Misorientation angle / °')
ax.set_aspect(aspect=100)
ax.grid()
plt.tight_layout()

Figure 3 - grain boundary sliding in early deformation

In [172]:
# Plot the ipf-x map for this zoomed in region


ebsd_map.plot_ipf_map([1,0,0],plot_gbs='line',dilate_boundaries=True,plot_scale_bar=True)
ax = plt.gca()
ax.set_xlim([1660,2140])
ax.set_ylim([1860,1390])

plt.savefig('figures_for_paper/Figure_3/ipf_x_zoom.png',dpi=1000)

In [ ]:
# let's find some areas where we've got some gb sliding early on 

# indices of steps we want to use
step_idxs = [2,3,4,7,10]

# number of steps we want to consider 
num_steps = len(step_idxs)

# range on maps
xrange = [1000,2000]
yrange = [2000,1000]
ems_crange = [0,0.02]
r_ang_crange = [-1,1]

# make figure
fig = plt.figure()
fig.set_size_inches([len(step_idxs)*2,4])
gs = gridspec.GridSpec(2,num_steps+1,width_ratios=np.append(np.ones((1,num_steps)),0.05))


for col_idx,step_idx in enumerate(step_idxs):

    ax0 = fig.add_subplot(gs[0,col_idx])
    ems_colours = ax0.imshow(ems_cube[:,:,step_idx],
            vmin=ems_crange[0],
            vmax=ems_crange[1])

    ax1 = fig.add_subplot(gs[1,col_idx])
    r_ang_colours = ax1.imshow(r_ang_cube[:,:,step_idx],
                vmin=r_ang_crange[0],
                vmax=r_ang_crange[1],
                cmap='RdBu_r')
    
    for ax in [ax0,ax1]:
        ax.set_xlim(xrange)
        ax.set_ylim(yrange)
        ax.set_xticks([])
        ax.set_yticks([])

    # get global x strain for title 
    glob_e11 = e11_glob[step_idx]*100

    ax0.set_title('$ϵ_{xx}$ = '+str(glob_e11.round(2))+'%')

# colourbars 
ax0 = fig.add_subplot(gs[0,-1])
fig.colorbar(ems_colours,cax=ax0,ticks=[ems_crange[0],np.mean(ems_crange),ems_crange[1]],label='Effective strain / -')

ax1 = fig.add_subplot(gs[1,-1])
fig.colorbar(r_ang_colours,cax=ax1,ticks=[r_ang_crange[0],0,r_ang_crange[1]],label='In-plane rotation / °')


plt.tight_layout()
plt.savefig('figures_for_paper/Figure_3/gb_sliding_progression.png',dpi=1000)

In [55]:
# mask thing by gb vs non-gb etc 

ebsd_map.data['grain_boundaries']

dilation_factor = 10
cleaning_footprint = 3
# dilate normal boundaries 
gbs = ski.morphology.binary_dilation(dic_map.data['grain_boundaries'].image,np.ones((cleaning_footprint,cleaning_footprint)))
sbs = ski.morphology.binary_dilation(dic_map.warp_to_dic_frame(s_bounds.image),np.ones((cleaning_footprint,cleaning_footprint)))

# xor to get other boundaries
ogbs = gbs ^ sbs

# boundary opening to destroy any tiny bits left 
ogbs = ski.morphology.binary_opening(ogbs,np.ones((cleaning_footprint,cleaning_footprint)))

# final dilation 
gbs     = ski.morphology.binary_dilation(gbs,np.ones((dilation_factor,dilation_factor)))
sbs     = ski.morphology.binary_dilation(sbs,np.ones((dilation_factor,dilation_factor)))
ogbs    = ski.morphology.binary_dilation(ogbs,np.ones((dilation_factor,dilation_factor)))

ogbs = ski.morphology.binary_opening(ogbs,np.ones((dilation_factor,dilation_factor)))

plt.figure()
plt.imshow(gbs)
plt.figure()
plt.imshow(sbs)

plt.figure()
plt.imshow(ogbs)


C:\Ben\Work\DefDAP\defdap\experiment.py:67: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `PolynomialTransform.from_estimate` class constructor instead.
  transform.estimate(
C:\Ben\Work\DefDAP\defdap\experiment.py:138: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mph.remove_small_objects(points_img, min_size=10, connectivity=2,
C:\Users\bepoole\AppData\Local\Temp\ipykernel_21144\29105413.py:8: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footpri

In [143]:
plt.figure()
plt.imshow(ems_cube[:,:,29],vmin=0,vmax=0.1)

In [144]:
# mask 
for i in range(0,ems_cube.shape[-1]):
    tdata = copy.deepcopy(ems_cube[:,:,i])

    ems_gbs = tdata[gbs]
    ems_sbs = tdata[sbs]
    ems_ogbs = tdata[ogbs]
    ems_gcs = tdata[np.invert(gbs)]

    logbins = np.logspace(-2,10,500)

    # fig,ax = plt.subplots(4,1,sharex=True,sharey=True)
    # ax[0].hist(ems_gbs,logbins,density=True,histtype='step')
    # ax[1].hist(ems_sbs,logbins,density=True,histtype='step')
    # ax[2].hist(ems_ogbs,logbins,density=True,histtype='step')
    # ax[3].hist(ems_gcs,logbins,density=True,histtype='step')

    # plt.xscale('log')
    # plt.yscale('log')

    fig,ax = plt.subplots()
    ax.hist(ems_gbs,logbins,density=True,histtype='step')
    ax.hist(ems_sbs,logbins,density=True,histtype='step')
    ax.hist(ems_ogbs,logbins,density=True,histtype='step')
    ax.hist(ems_gcs,logbins,density=True,histtype='step')

    plt.legend(['All GBs','S3GBs', 'Other GBs','Grain cores'])

    plt.xscale('log')
    plt.yscale('log')
    ax.set_xlim(1e-2,10)
    ax.set_ylim(1e-2,1e3)
    ax.set_xlabel('Max shear strain / -')
    ax.set_ylabel('Probability density / -')

    plt.savefig('ems_hist_step_' +str(i)+'.jpeg')
    plt.close(fig)
    